#### Check before running

In [1]:
%%sql
select * from lh_vv_gold.dbo.ai_eventhub_sync_config

StatementMeta(, f327bcdb-b5e0-4259-b993-9905622f7f53, 2, Finished, Available, Finished, False)

<Spark SQL result set with 14 rows and 15 fields>

#### Modify if needed

In [2]:
%%sql
SELECT * FROM lh_vv_gold.dbo.ai_eventhub_watermark
--DELETE FROM lh_vv_gold.dbo.ai_eventhub_watermark
-- delete from lh_vv_gold.dbo.ai_eventhub_watermark where watermark_id = 'lh_vv_gold.dbo.ai_product'
--delete from lh_vv_gold.dbo.ai_eventhub_watermark where watermark_id = 'lh_vv_gold.dbo.ai_enriched_poi_prd_clone'

StatementMeta(, f327bcdb-b5e0-4259-b993-9905622f7f53, 3, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 8 fields>

#### Run this session to send event stream
test mode will only send POI from partner portal

In [3]:
# ============================================================
# Notebook: Config-driven Table -> Event Hub Publisher
# Version:
#   - No CREATE TABLE scripts
#   - Watermark table uses watermark_id only
#   - Source payload columns are cast to STRING for stability
#   - Spark datetime parser policy = LEGACY
#   - Spark ANSI disabled
#   - Support tables WITHOUT watermark_col as FULL_REFRESH mode
#   - Independent table processing
#   - Record-level error isolation
#   - Invalid watermark datetime handling
#   - Event Hub throttling retry
#   - Event Hub global limiter by:
#       + events / second
#       + bytes / second
#       + events / batch
#       + bytes / batch
#   - Optional serialized Event Hub send
#   - Retry Delta writes for audit / run audit / watermark
#   - Serialize metadata Delta writes with threading.Lock
#   - Do NOT raise exception at the end if some tables fail
#
# Payload format is unchanged:
# {
#   "schema_version": "1",
#   "event_id": "<uuid>",
#   "event_timestamp": "<utc timestamp>",
#   "source": "fabric",
#   "entity_type": "<entity_type>",
#   "entity_id": "<entity_id>",
#   "event_type": "upsert",
#   "data": { full source row, all values as string/null }
# }
# ============================================================

import json
import time
import uuid
import base64
import random
import threading
import traceback
import os
import re
from datetime import datetime, date, timezone
from decimal import Decimal
from concurrent.futures import ThreadPoolExecutor, as_completed

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,


    LongType,
    NumericType,
    BooleanType
)

# ============================================================
# 0. SPARK CONFIG
# ============================================================

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.ansi.enabled", "false")


# ============================================================
# 1. GLOBAL CONFIG
# ============================================================

VARIABLE_LIBRARY = "dev_variables"

CONFIG_TABLE = "lh_vv_gold.dbo.ai_eventhub_sync_config"
WATERMARK_TABLE = "lh_vv_gold.dbo.ai_eventhub_watermark"
AUDIT_TABLE = "lh_vv_gold.dbo.ai_eventhub_push_audit"
RUN_AUDIT_TABLE = "lh_vv_gold.dbo.ai_eventhub_run_audit"

DEFAULT_BATCH_SIZE = 1000 # Change the number for testing mode

INGESTION_TARGET = "all" # all | memgraph | qdrant | pg_identity


TEST_MODE = False
TEST_ROWS_PER_TABLE = 10

# Default mode
ROW_FILTER = {
    "ai_enriched_poi": "ingestion_source = 'partner_portal'", # google | partner_portal
}

# Using when new POI is added today
#ROW_FILTER = {
#    "ai_enriched_poi": """
#        ingestion_source = 'google'
#        AND updated_at >= current_date()
#    """
#}


TEST_ROW_FILTER = {
    "ai_enriched_poi": "ingestion_source = 'google'", # google | partner_portal
}
    

ENTITY_TYPE_MAP = {
    'ai_enriched_poi': 'poi',
    'ai_product': 'product',
    'ai_destination': 'destination',
    'destination': 'destination',
    'business_services': 'poi',   # partner venues -> POI nodes
    'product_posts': 'product',
}

def canonical_entity_type(value):
    key = (str(value).strip().lower() if value is not None else '')
    return ENTITY_TYPE_MAP.get(key, key)

# ============================================================
# KB FIELD CONTRACT
# Align each payload's `data` dict with what knowledge-base-ingestion /
# Memgraph search expects, keyed by CANONICAL entity_type. Applied AFTER JSON
# nesting (apply_kb_field_contract), so dict/list columns such as
# operating_hours / *_ids are real objects here. Every step is a no-op when its
# source field is absent, so it is safe across heterogeneous source tables that
# map to the same entity_type. Only fields whose fix-location is EVENT (the
# push notebook) are handled here; INGEST-only and "add a field the source does
# not have" fixes are intentionally NOT done here.
# ============================================================

# 1. Source-column -> KB-property renames (the source column is renamed in place).
ENTITY_FIELD_RENAMES = {
    'poi': {
        'poi_name': 'name',                          # POIDetail.name (required)
        'ref_time_to_spend_minutes': 'ref_time_spend',
        'ref_cost_vnd': 'ref_cost',
    },
    'offer': {
        'promotion_start': 'apply_date_from',
        'promotion_end': 'apply_date_to',
    },
}


# 2. KB MATCH-key aliases: search Cypher matches on <x>_id, but the event only
#    carries the uniform `id`. Copy id -> the entity's match key (id is kept too).
ENTITY_ID_ALIASES = {
    'offer': 'offer_id',
    'blog': 'blog_id',
    'news': 'news_id',
    'faq': 'doc_id',
}

# 3. Per-language id dicts that ingestion drops (nested dict) -> flatten into two
#    top-level scalar fields the search layer reads.
ENTITY_DICT_FLATTEN = {
    'blog': {'source': 'blog_ids', 'vi': 'blog_id_vi', 'en': 'blog_id_en'},
    'news': {'source': 'news_ids', 'vi': 'news_id_vi', 'en': 'news_id_en'},
}

# 4. Nested columns ingestion silently drops (only scalars are stored) ->
#    serialize to a JSON string so the value survives as a scalar property.
ENTITY_JSON_STRINGIFY = {
    'poi': ('operating_hours',),
}

# 5. Required name fields that must be non-null. Value is an ordered coalesce:
#    the first truthy field in the list wins (so for offers promotion_title is
#    preferred over name, matching `name = promotion_title or name`).
ENTITY_NAME_COALESCE = {
    'offer': {'name': ('promotion_title', 'name')},
}


def clean_subcategory_tags(value):
    """Normalize ai_enriched_poi.subcategory_tags into a clean list of tags.
    Upstream the column arrives as an array whose first/last elements still
    carry the literal [ and ] from a stringified list, e.g.
    ["[amusement_park", "cafe", "theme_park]"], or sometimes as one bracketed
    string. Strip stray brackets/quotes/whitespace from each tag, drop blanks.
    Returns a list (possibly empty) or None when there is nothing."""
    if value is None:
        return None

    # A single string -> prefer real JSON, else fall back to comma split.
    if isinstance(value, str):
        s = value.strip()
        if not s:
            return None
        try:
            parsed = json.loads(s)
        except (ValueError, TypeError):
            parsed = None
        value = parsed if isinstance(parsed, list) else s.split(",")

    if not isinstance(value, (list, tuple)):
        return value

    cleaned = []
    for item in value:
        if item is None:
            continue
        tag = str(item).strip().strip("[]").strip().strip("\"'").strip()
        if tag:
            cleaned.append(tag)

    return cleaned or None



def _coerce_to_dict(value):
    # Accept either a real dict (post-nesting) or a JSON-object string.
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        s = value.strip()
        if s.startswith('{') and s.endswith('}'):
            try:
                parsed = json.loads(s)
            except (ValueError, TypeError):
                return None
            if isinstance(parsed, dict):
                return parsed
    return None


def apply_kb_field_contract(data: dict, entity_type) -> dict:
    """Mutate a payload `data` dict in place to satisfy the Memgraph KB field
    contract for the given canonical entity_type. Safe to call for any entity;
    each step only fires when its source field is present."""
    if not isinstance(data, dict):
        return data

    et = (str(entity_type).strip().lower() if entity_type is not None else '')

    # 1. renames
    for src_col, dst_col in ENTITY_FIELD_RENAMES.get(et, {}).items():
        if src_col in data:
            data[dst_col] = data.pop(src_col)

    # 2. id -> match-key alias
    alias = ENTITY_ID_ALIASES.get(et)
    if alias and data.get('id') is not None:
        data[alias] = data['id']

    # 3. per-language id flatten
    flat = ENTITY_DICT_FLATTEN.get(et)
    if flat:
        d = _coerce_to_dict(data.get(flat['source']))
        if d is not None:
            data[flat['vi']] = d.get('vi')
            data[flat['en']] = d.get('en')
            data.pop(flat['source'], None)

    # 4. nested dict/list -> JSON string scalar
    for col in ENTITY_JSON_STRINGIFY.get(et, ()):
        v = data.get(col)
        if isinstance(v, (dict, list)):
            data[col] = json_dumps_compact(v)

    # 5. required-name coalesce (first truthy source wins)
    for target, sources in ENTITY_NAME_COALESCE.get(et, {}).items():
        for s in sources:
            if data.get(s):
                data[target] = data[s]
                break

    # 6. POI only: strip stray [ ] / quotes left on subcategory_tags elements.
    if et == 'poi' and 'subcategory_tags' in data:
        data['subcategory_tags'] = clean_subcategory_tags(data['subcategory_tags'])

    return data

# --- Language filter ------------------------------------------------------
# Entity types that must publish English rows only. Applied on EVERY run
# (not just TEST_MODE) as an equality filter on the source table before
# payload build. Source tables for these entity types must expose
# LANG_FILTER_COLUMN; a missing column is treated as a config error.
LANG_FILTER_ENTITY_TYPES = {'product', 'poi', 'destination'}
LANG_FILTER_COLUMN = 'lang'
LANG_FILTER_VALUE = 'en'
ACTIVE_FILTER_COLUMN = 'is_active'
ACTIVE_FILTER_VALUE = True

# --- Soft-delete on is_active=false (ai_enriched_poi only) ----------------
# Normally rows with is_active=false are dropped entirely by ACTIVE_FILTER.
# For this config only, an inactive row may be published as a soft_delete event
# if it looks like a previously synced row that was deactivated after the last
# successful run:
#   - created_at <= watermark AND updated_at > watermark -> send soft_delete
#   - created_at >  watermark                            -> skip; it was never synced
#   - updated_at <= watermark                            -> skip; deactivation is not new
# When no watermark has been recorded yet for the config (first run), all
# inactive rows are skipped because nothing has been synced downstream yet.

POI_SOFT_DELETE_CONFIG_IDS = {
    "cfg_lh_vv_gold_dbo_ai_enriched_poi"
}

PRODUCT_SOFT_DELETE_CONFIG_IDS = {
    "cfg_lh_vv_gold_dbo_ai_product"
}

SOFT_DELETE_CONFIG_IDS = (
    POI_SOFT_DELETE_CONFIG_IDS
    | PRODUCT_SOFT_DELETE_CONFIG_IDS
)
SOFT_DELETE_CREATED_DATE_COLUMN = 'created_at'
SOFT_DELETE_EVENT_TYPE = 'soft_delete'

# --- Destination ID remap (KB Memgraph ids differ from source ids) -------
# A handful of destination entities were created in Memgraph with different
# node ids than their source destination_id. Remap destination_id (whether
# it is the row's own id_key or a foreign-key reference to the destination
# entity) before payload build. Only applied for the two configs below.
DESTINATION_ID_REMAP_CONFIG_IDS = {
    'cfg_lh_vv_gold_dbo_ai_enriched_poi',
    'cfg_lh_vv_gold_dbo_destination',
}
DESTINATION_ID_REMAP = {
    'npl86zvhuytfb0ys679uz6w1': 'fc11vukvpvi6dxojzymmwfl4',  # Bana Hill
    'w9w1hdm8iwcofbh1rj12a3i2': 'chdm4mekx35eqiiscri33t30',  # Hue
    'bpy3b4acd6hjdisugue3sxz7': 'e8xuqhikk4547xjw4haffqko',  # Da Nang
    't36nuvkacqp87p6pk1k4zbv8': 'xjo4p6jhlvabpjhfyq4rgac7',  # Hoi An
}


def remap_destination_id(row_dict: dict, config: dict) -> None:
    """Mutate row_dict in place: remap destination_id to its KB-side id for
    the configs where the two id spaces diverge. No-op if destination_id is
    absent or has no configured remap."""
    if config.get("config_id") not in DESTINATION_ID_REMAP_CONFIG_IDS:
        return
    old_id = row_dict.get("destination_id")
    if old_id is None:
        return
    new_id = DESTINATION_ID_REMAP.get(str(old_id).strip())
    if new_id is not None:
        row_dict["destination_id"] = new_id

# --- Required non-null columns -------------------------------------------
# Rows whose listed column(s) are NULL (or blank after trim) are dropped before
# payload build, keyed by canonical entity_type. Applied on EVERY run (not just
# TEST_MODE). A listed column missing from the source table is a config error.
REQUIRED_NON_NULL_COLUMNS = {
    'product': ['source_id'],
}

# --- Required non-null columns, scoped to a specific config_id ------------
# Same idea as REQUIRED_NON_NULL_COLUMNS above, but keyed by config_id instead
# of entity_type. Use this when the rule only applies to one source table
# (e.g. only ai_enriched_poi must have destination_id set) rather than to
# every table that maps to a given canonical entity_type (poi also covers
# business_services, which has no destination_id column).
REQUIRED_NON_NULL_COLUMNS_BY_CONFIG = {
    'cfg_lh_vv_gold_dbo_ai_enriched_poi': ['destination_id'],  # POI must have a destination to publish
}

# Safer defaults for throttled Event Hub Standard.
# Có thể override bằng Variable Library hoặc spark.conf.
DEFAULT_MAX_EVENTS_PER_SECOND = 200
DEFAULT_MAX_PARALLEL_TABLES = 2

DEFAULT_EH_MAX_EVENTS_PER_BATCH = 500
DEFAULT_EH_MAX_BYTES_PER_BATCH = 128_000
DEFAULT_EH_MAX_BYTES_PER_SECOND = 256_000
DEFAULT_EH_SERIALIZE_SEND = True

DEFAULT_PAYLOAD_SOFT_LIMIT_BYTES = 950_000

DEFAULT_DELTA_WRITE_MAX_RETRIES = 8
DEFAULT_DELTA_WRITE_RETRY_BASE_SECONDS = 3

DEFAULT_EH_SEND_MAX_RETRIES = 12
DEFAULT_EH_SEND_BASE_WAIT_SECONDS = 8

DEBUG = False

NO_WATERMARK_VALUES = {
    "",
    "none",
    "null",
    "n/a",
    "na",
    "no",
    "không có",
    "khong co",
    "không_co",
    "khong_co",
    "no watermark",
    "no_watermark"
}


# ============================================================
# 2. VARIABLE / SECRET HELPER - FABRIC SAFE VERSION
# ============================================================

def _try_get_notebookutils():
    try:
        return notebookutils
    except Exception:
        return None


def _try_get_mssparkutils():
    try:
        return mssparkutils
    except Exception:
        return None


def _value_is_empty(value) -> bool:
    return value is None or str(value).strip() == ""


def _extract_value_from_library_object(vl, var_name: str):
    value = None
    errors = []

    if vl is None:
        return None, ["Variable library object is None"]

    try:
        value = vl.getVariable(var_name)
        if not _value_is_empty(value):
            return value, errors
    except Exception as e:
        errors.append(f"getVariable({var_name}): {e}")

    try:
        value = vl.get(var_name)
        if not _value_is_empty(value):
            return value, errors
    except Exception as e:
        errors.append(f"get({var_name}): {e}")

    try:
        value = getattr(vl, var_name)
        if not _value_is_empty(value):
            return value, errors
    except Exception as e:
        errors.append(f"getattr({var_name}): {e}")

    try:
        if isinstance(vl, dict):
            value = vl.get(var_name)
            if not _value_is_empty(value):
                return value, errors
    except Exception as e:
        errors.append(f"dict.get({var_name}): {e}")

    return None, errors


def _vl_get(lib_name: str, var_name: str, default=None, required: bool = False):
    value = None
    errors = []

    nb_utils = _try_get_notebookutils()
    ms_utils = _try_get_mssparkutils()

    if value is None and nb_utils is not None:
        try:
            vl = nb_utils.variableLibrary.getLibrary(lib_name)
            value, sub_errors = _extract_value_from_library_object(vl, var_name)
            errors.extend([f"notebookutils.getLibrary object: {e}" for e in sub_errors])
        except Exception as e:
            errors.append(f"notebookutils.variableLibrary.getLibrary({lib_name}): {e}")

    if _value_is_empty(value) and nb_utils is not None:
        try:
            ref = f"$(/**/{lib_name}/{var_name})"
            value = nb_utils.variableLibrary.get(ref)
        except Exception as e:
            errors.append(f"notebookutils.variableLibrary.get({ref}): {e}")

    if _value_is_empty(value) and nb_utils is not None:
        try:
            value = nb_utils.variableLibrary.get(var_name)
        except Exception as e:
            errors.append(f"notebookutils.variableLibrary.get({var_name}): {e}")

    if _value_is_empty(value) and nb_utils is not None:
        try:
            value = nb_utils.variableLibrary.get(f"{lib_name}.{var_name}")
        except Exception as e:
            errors.append(f"notebookutils.variableLibrary.get({lib_name}.{var_name}): {e}")

    if _value_is_empty(value) and ms_utils is not None:
        try:
            vl = ms_utils.variableLibrary.getLibrary(lib_name)
            value, sub_errors = _extract_value_from_library_object(vl, var_name)
            errors.extend([f"mssparkutils.getLibrary object: {e}" for e in sub_errors])
        except Exception as e:
            errors.append(f"mssparkutils.variableLibrary.getLibrary({lib_name}): {e}")

    if _value_is_empty(value) and ms_utils is not None:
        try:
            ref = f"$(/**/{lib_name}/{var_name})"
            value = ms_utils.variableLibrary.get(ref)
        except Exception as e:
            errors.append(f"mssparkutils.variableLibrary.get({ref}): {e}")

    if _value_is_empty(value) and ms_utils is not None:
        try:
            value = ms_utils.variableLibrary.get(var_name)
        except Exception as e:
            errors.append(f"mssparkutils.variableLibrary.get({var_name}): {e}")

    if _value_is_empty(value):
        try:
            value = spark.conf.get(var_name)
        except Exception as e:
            errors.append(f"spark.conf.get({var_name}): {e}")

    if _value_is_empty(value):
        try:
            value = spark.conf.get(f"{lib_name}.{var_name}")
        except Exception as e:
            errors.append(f"spark.conf.get({lib_name}.{var_name}): {e}")

    if _value_is_empty(value):
        try:
            value = os.getenv(var_name)
        except Exception as e:
            errors.append(f"os.getenv({var_name}): {e}")

    if _value_is_empty(value):
        try:
            value = os.getenv(f"{lib_name}_{var_name}")
        except Exception as e:
            errors.append(f"os.getenv({lib_name}_{var_name}): {e}")

    if _value_is_empty(value):
        if required:
            raise Exception(
                f"Missing required variable '{var_name}' in Variable Library '{lib_name}'. "
                f"Tried notebookutils.variableLibrary, mssparkutils.variableLibrary, spark.conf, and os.environ. "
                f"Recent errors: {' | '.join(errors[-10:])}"
            )
        return default

    return str(value).strip()


# ============================================================
# 3. LOAD VARIABLES
# ============================================================

EH_NAMESPACE = _vl_get(VARIABLE_LIBRARY, "ai_eventhub_namespace", required=True)
EH_NAME = _vl_get(VARIABLE_LIBRARY, "ai_eventhub_name", required=True)
EH_POLICY = _vl_get(VARIABLE_LIBRARY, "ai_eventhub_policy", required=True)
EH_KEY = _vl_get(VARIABLE_LIBRARY, "ai_eventhub_key", required=True)

MAX_EVENTS_PER_SECOND = int(_vl_get(
    VARIABLE_LIBRARY,
    "EH_MAX_EVENTS_PER_SECOND",
    DEFAULT_MAX_EVENTS_PER_SECOND
))

MAX_PARALLEL_TABLES = int(_vl_get(
    VARIABLE_LIBRARY,
    "EH_MAX_PARALLEL_TABLES",
    DEFAULT_MAX_PARALLEL_TABLES
))

EH_MAX_EVENTS_PER_BATCH = int(_vl_get(
    VARIABLE_LIBRARY,
    "EH_MAX_EVENTS_PER_BATCH",
    DEFAULT_EH_MAX_EVENTS_PER_BATCH
))

EH_MAX_BYTES_PER_BATCH = int(_vl_get(
    VARIABLE_LIBRARY,
    "EH_MAX_BYTES_PER_BATCH",
    DEFAULT_EH_MAX_BYTES_PER_BATCH
))

EH_MAX_BYTES_PER_SECOND = int(_vl_get(
    VARIABLE_LIBRARY,
    "EH_MAX_BYTES_PER_SECOND",
    DEFAULT_EH_MAX_BYTES_PER_SECOND
))

EH_SERIALIZE_SEND = str(_vl_get(
    VARIABLE_LIBRARY,
    "EH_SERIALIZE_SEND",
    DEFAULT_EH_SERIALIZE_SEND
)).lower() in ["true", "1", "yes", "y"]

PAYLOAD_SOFT_LIMIT_BYTES = int(_vl_get(
    VARIABLE_LIBRARY,
    "EH_PAYLOAD_SOFT_LIMIT_BYTES",
    DEFAULT_PAYLOAD_SOFT_LIMIT_BYTES
))

GLOBAL_DEFAULT_BATCH_SIZE = int(_vl_get(
    VARIABLE_LIBRARY,
    "DEFAULT_BATCH_SIZE",
    DEFAULT_BATCH_SIZE
))

def _vl_bool(key, default):
    v = _vl_get(key, None)
    if v is None: return default
    return str(v).strip().lower() in ('1','true','yes','t','y')

TEST_MODE = _vl_bool('TEST_MODE', TEST_MODE)


DELTA_WRITE_MAX_RETRIES = int(_vl_get(
    VARIABLE_LIBRARY,
    "DELTA_WRITE_MAX_RETRIES",
    DEFAULT_DELTA_WRITE_MAX_RETRIES
))

DELTA_WRITE_RETRY_BASE_SECONDS = int(_vl_get(
    VARIABLE_LIBRARY,
    "DELTA_WRITE_RETRY_BASE_SECONDS",
    DEFAULT_DELTA_WRITE_RETRY_BASE_SECONDS
))

EH_SEND_MAX_RETRIES = int(_vl_get(
    VARIABLE_LIBRARY,
    "EH_SEND_MAX_RETRIES",
    DEFAULT_EH_SEND_MAX_RETRIES
))

EH_SEND_BASE_WAIT_SECONDS = int(_vl_get(
    VARIABLE_LIBRARY,
    "EH_SEND_BASE_WAIT_SECONDS",
    DEFAULT_EH_SEND_BASE_WAIT_SECONDS
))

if DEBUG:
    print("========== EVENT HUB CONFIG ==========")
    print(f"EH_NAMESPACE                    = {EH_NAMESPACE}")
    print(f"EH_NAME                         = {EH_NAME}")
    print(f"EH_POLICY                       = {EH_POLICY}")
    print(f"MAX_EVENTS_PER_SECOND_GLOBAL    = {MAX_EVENTS_PER_SECOND}")
    print(f"MAX_PARALLEL_TABLES             = {MAX_PARALLEL_TABLES}")
    print(f"EH_MAX_EVENTS_PER_BATCH         = {EH_MAX_EVENTS_PER_BATCH}")
    print(f"EH_MAX_BYTES_PER_BATCH          = {EH_MAX_BYTES_PER_BATCH}")
    print(f"EH_MAX_BYTES_PER_SECOND         = {EH_MAX_BYTES_PER_SECOND}")
    print(f"EH_SERIALIZE_SEND               = {EH_SERIALIZE_SEND}")
    print(f"PAYLOAD_SOFT_LIMIT_BYTES        = {PAYLOAD_SOFT_LIMIT_BYTES}")
    print(f"GLOBAL_DEFAULT_BATCH_SIZE       = {GLOBAL_DEFAULT_BATCH_SIZE}")
    print(f"DELTA_WRITE_MAX_RETRIES         = {DELTA_WRITE_MAX_RETRIES}")
    print(f"DELTA_WRITE_RETRY_BASE_SECONDS  = {DELTA_WRITE_RETRY_BASE_SECONDS}")
    print(f"EH_SEND_MAX_RETRIES             = {EH_SEND_MAX_RETRIES}")
    print(f"EH_SEND_BASE_WAIT_SECONDS       = {EH_SEND_BASE_WAIT_SECONDS}")
    print("======================================")


# ============================================================
# 4. INSTALL / IMPORT EVENT HUB SDK
# ============================================================

try:
    from azure.eventhub import EventHubProducerClient, EventData
except Exception:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "azure-eventhub"])
    from azure.eventhub import EventHubProducerClient, EventData


def build_eventhub_connection_string() -> str:
    return (
        f"Endpoint=sb://{EH_NAMESPACE}.servicebus.windows.net/;"
        f"SharedAccessKeyName={EH_POLICY};"
        f"SharedAccessKey={EH_KEY};"
    )


EH_CONNECTION_STRING = build_eventhub_connection_string()


# ============================================================
# 5. GLOBAL RATE LIMITER + LOCKS
# ============================================================

class GlobalRateLimiter:
    """
    Global limiter by events/second and bytes/second.
    This protects Event Hub from IncomingBytes throttling.
    """

    def __init__(self, max_events_per_second: int, max_bytes_per_second: int):
        self.max_events_per_second = max_events_per_second
        self.max_bytes_per_second = max_bytes_per_second

        self.lock = threading.Lock()
        self.window_start = time.time()
        self.events_in_window = 0
        self.bytes_in_window = 0

    def acquire(self, events_to_send: int, bytes_to_send: int):
        if self.max_events_per_second <= 0 and self.max_bytes_per_second <= 0:
            return

        events_needed = max(int(events_to_send), 1)
        bytes_needed = max(int(bytes_to_send), 1)

        with self.lock:
            while True:
                now = time.time()
                elapsed = now - self.window_start

                if elapsed >= 1:
                    self.window_start = now
                    self.events_in_window = 0
                    self.bytes_in_window = 0
                    elapsed = 0

                events_available = (
                    self.max_events_per_second - self.events_in_window
                    if self.max_events_per_second > 0
                    else events_needed
                )

                bytes_available = (
                    self.max_bytes_per_second - self.bytes_in_window
                    if self.max_bytes_per_second > 0
                    else bytes_needed
                )

                # Nếu payload/batch lớn hơn limit mỗi giây, vẫn cho gửi sau khi window trống.
                # Điều này tránh deadlock khi bytes_needed > max_bytes_per_second.
                events_ok = events_available >= min(events_needed, self.max_events_per_second) if self.max_events_per_second > 0 else True
                bytes_ok = bytes_available >= min(bytes_needed, self.max_bytes_per_second) if self.max_bytes_per_second > 0 else True

                if events_ok and bytes_ok:
                    self.events_in_window += events_needed
                    self.bytes_in_window += bytes_needed
                    return

                sleep_sec = max(0.05, 1 - elapsed)
                time.sleep(sleep_sec)


GLOBAL_EH_RATE_LIMITER = GlobalRateLimiter(
    max_events_per_second=MAX_EVENTS_PER_SECOND,
    max_bytes_per_second=EH_MAX_BYTES_PER_SECOND
)

EVENTHUB_SEND_LOCK = threading.Lock()

WATERMARK_WRITE_LOCK = threading.Lock()
PUSH_AUDIT_WRITE_LOCK = threading.Lock()
RUN_AUDIT_WRITE_LOCK = threading.Lock()


# ============================================================
# 6. COMMON HELPERS
# ============================================================

def utc_now():
    return datetime.now(timezone.utc)


def utc_now_iso():
    return utc_now().strftime("%Y-%m-%dT%H:%M:%SZ")


def sql_quote(value) -> str:
    if value is None:
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"


def quote_identifier(col_name: str) -> str:
    return "`" + str(col_name).replace("`", "``") + "`"


def normalize_text(value):
    if value is None:
        return ""
    return str(value).strip()


def normalize_lower(value):
    return normalize_text(value).lower()


def build_table_full_name(layer_name: str, table_schema: str, table_name: str) -> str:
    return f"{layer_name}.{table_schema}.{table_name}"


def build_watermark_id(config: dict) -> str:
    wm_id = config.get("watermark_id")
    if wm_id is not None and str(wm_id).strip() != "":
        return str(wm_id).strip()

    return f"{config['layer_name']}.{config['table_schema']}.{config['table_name']}"


def has_watermark(config: dict) -> bool:
    wm_col = normalize_lower(config.get("watermark_col"))
    return wm_col not in NO_WATERMARK_VALUES


def get_watermark_col(config: dict):
    if not has_watermark(config):
        return None
    return normalize_text(config.get("watermark_col"))


def get_processing_mode(config: dict) -> str:
    return "INCREMENTAL" if has_watermark(config) else "FULL_REFRESH"


def normalize_select_columns(select_columns: str):
    if select_columns is None:
        return None

    raw = str(select_columns).strip()
    if raw == "" or raw.lower() in ["all", "all column", "all columns", "*"]:
        return None

    return [c.strip() for c in raw.split(",") if c.strip()]


def _payload_select_exprs(df, columns):
    # Schema-aware projection for payloads. Numeric and boolean columns keep
    # their native Spark type so they serialize as real JSON numbers/booleans;
    # every other type (string, date, timestamp, struct, array, binary) is cast
    # to STRING, preserving the original "stable string" behavior. Driven by the
    # SOURCE schema, so genuine string columns (phone, ids, country_code) stay
    # strings — no value guessing.
    types = {f.name: f.dataType for f in df.schema}
    exprs = []
    for c in columns:
        dt = types.get(c)
        if isinstance(dt, (NumericType, BooleanType)):
            exprs.append(F.col(c))                       # native -> JSON number/bool
        else:
            exprs.append(F.col(c).cast("string").alias(c))
    return exprs


def safe_json_value(v):
    if v is None:
        return None

    # Keep native scalars so they serialize as real JSON number/bool, not quoted
    # strings. bool must be checked before int (bool is a subclass of int).
    if isinstance(v, bool):
        return v

    if isinstance(v, (int, float)):
        # NaN / +-Infinity are not valid JSON -> emit null instead.
        if isinstance(v, float) and (v != v or v == float("inf") or v == float("-inf")):
            return None
        return v

    if isinstance(v, Decimal):
        return float(v)   # emit decimals as JSON numbers

    if isinstance(v, datetime):
        if v.tzinfo is None:
            return v.replace(tzinfo=timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
        return v.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

    if isinstance(v, date):
        return v.isoformat()

    if isinstance(v, (bytes, bytearray)):
        return base64.b64encode(v).decode("utf-8")

    return str(v)


def _to_nested_json_value(value):
    # Turn JSON-OBJECT and JSON-ARRAY strings into real nested JSON so the
    # payload carries structured objects/arrays instead of escaped strings.
    # Recurses so deeply-nested JSON strings are also expanded. Any string
    # that is not valid JSON (e.g. Spark array casts like "[a, b]") fails
    # json.loads and is returned unchanged.
    if isinstance(value, dict):
        return {k: _to_nested_json_value(v) for k, v in value.items()}

    if isinstance(value, list):
        return [_to_nested_json_value(v) for v in value]

    if isinstance(value, str):
        s = value.strip()
        if (s.startswith("{") and s.endswith("}")) or (s.startswith("[") and s.endswith("]")):
            try:
                parsed = json.loads(s)
            except (ValueError, TypeError):
                return value
            if isinstance(parsed, (dict, list)):
                return _to_nested_json_value(parsed)
        return value

    return value


def row_to_json_dict(row_dict: dict) -> dict:
    return {k: _to_nested_json_value(safe_json_value(v)) for k, v in row_dict.items()}


def json_dumps_compact(obj: dict) -> str:
    return json.dumps(obj, ensure_ascii=False, separators=(",", ":"), default=str)


def payload_size_bytes(payload_json: str) -> int:
    return len(payload_json.encode("utf-8"))


def build_entity_id(row_dict: dict, id_key: str) -> str:
    keys = [k.strip() for k in str(id_key).split(",") if k.strip()]
    values = []

    for k in keys:
        v = row_dict.get(k)
        values.append("" if v is None else str(v))

    return "|".join(values)


StatementMeta(, f327bcdb-b5e0-4259-b993-9905622f7f53, 5, Finished, Available, Finished, False)

In [4]:
def build_payload(
    row_dict: dict,
    entity_type: str,
    entity_id: str,
    event_type: str,
    source_name: str
) -> dict:
    return {
        "schema_version": "1.0",
        "event_id": str(uuid.uuid4()),
        "event_timestamp": utc_now_iso(),
        "ingestion_target": INGESTION_TARGET,
        "source": source_name or "fabric",
        "entity_type": entity_type,
        "entity_id": entity_id,
        "event": event_type or "upsert", # match the schema of knowledge-base-ingestion
        "data": row_to_json_dict(row_dict)
    }


# ============================================================
# 7. RETRY HELPERS FOR DELTA WRITES
# ============================================================

def is_retryable_delta_error(err: Exception) -> bool:
    msg = str(err).lower()

    retry_keywords = [
        "delta_concurrent_append",
        "concurrentappendexception",
        "concurrent append",
        "concurrentappend",
        "concurrentdelete",
        "concurrent delete",
        "concurrentupdate",
        "concurrent update",
        "metadatachanged",
        "metadata changed",
        "files were added",
        "conflicting commit",
        "optimistictransaction",
        "transaction",
        "conflict",
        "timeout",
        "temporarily",
        "rate",
        "429",
        "503",
        "504"
    ]

    return any(k in msg for k in retry_keywords)


def with_retry(action_name: str, fn, max_retries: int = None, base_sleep: int = None):
    if max_retries is None:
        max_retries = DELTA_WRITE_MAX_RETRIES

    if base_sleep is None:
        base_sleep = DELTA_WRITE_RETRY_BASE_SECONDS

    last_err = None

    for attempt in range(1, max_retries + 1):
        try:
            return fn()
        except Exception as e:
            last_err = e

            if not is_retryable_delta_error(e) or attempt >= max_retries:
                raise

            sleep_sec = base_sleep * attempt + random.uniform(0, 2.5)

            if DEBUG:
                print(
                    f"[RETRY] action={action_name} "
                    f"attempt={attempt}/{max_retries} "
                    f"sleep={sleep_sec:.2f}s "
                    f"error={str(e)[:500]}"
                )

            time.sleep(sleep_sec)

    raise last_err


# ============================================================
# 8. EVENT HUB THROTTLING RETRY HELPERS
# ============================================================

def is_retryable_eventhub_error(err: Exception) -> bool:
    msg = str(err).lower()

    retry_keywords = [
        "throttled",
        "entity is being throttled",
        "server busy",
        "serverbusy",
        "servicebusserverbusy",
        "50002",
        "sub error : 105",
        "sub error: 105",
        "timeout",
        "temporarily",
        "please wait",
        "try again",
        "429",
        "503",
        "504"
    ]

    return any(k in msg for k in retry_keywords)


def extract_wait_seconds_from_eventhub_error(err: Exception, default_wait: int = None) -> int:
    if default_wait is None:
        default_wait = EH_SEND_BASE_WAIT_SECONDS

    msg = str(err)
    m = re.search(r"wait\s+(\d+)\s+seconds", msg, re.IGNORECASE)

    if m:
        return max(int(m.group(1)), default_wait)

    return default_wait


def send_batch_with_retry(producer, batch, event_count: int, byte_count: int):
    last_err = None

    for attempt in range(1, EH_SEND_MAX_RETRIES + 1):
        try:
            GLOBAL_EH_RATE_LIMITER.acquire(
                events_to_send=event_count,
                bytes_to_send=byte_count
            )

            if EH_SERIALIZE_SEND:
                with EVENTHUB_SEND_LOCK:
                    producer.send_batch(batch)
            else:
                producer.send_batch(batch)

            return

        except Exception as e:
            last_err = e

            if not is_retryable_eventhub_error(e) or attempt >= EH_SEND_MAX_RETRIES:
                raise

            wait_sec = extract_wait_seconds_from_eventhub_error(
                e,
                default_wait=EH_SEND_BASE_WAIT_SECONDS
            )

            msg = str(e).lower()
            incoming_bytes_penalty = 8 if "incomingbytes" in msg else 0

            sleep_sec = wait_sec + incoming_bytes_penalty + (attempt * 2) + random.uniform(0, 5)

            print(
                f"[EVENTHUB_RETRY] attempt={attempt}/{EH_SEND_MAX_RETRIES} "
                f"event_count={event_count} "
                f"byte_count={byte_count} "
                f"sleep={sleep_sec:.2f}s "
                f"error={str(e)[:1000]}"
            )

            time.sleep(sleep_sec)

    raise last_err


# ============================================================
# 9. AUDIT SCHEMAS + AUDIT WRITES
# ============================================================

PUSH_AUDIT_SCHEMA = StructType([
    StructField("audit_id", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("config_id", StringType(), True),
    StructField("layer_name", StringType(), True),
    StructField("table_schema", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("entity_type", StringType(), True),
    StructField("entity_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("failure_stage", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("payload_json", StringType(), True),
    StructField("record_watermark_value", TimestampType(), True),
    StructField("batch_id", StringType(), True),
    StructField("created_at", TimestampType(), True),
])

RUN_AUDIT_SCHEMA = StructType([
    StructField("run_audit_id", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("config_id", StringType(), True),
    StructField("layer_name", StringType(), True),
    StructField("table_schema", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("records_read", LongType(), True),
    StructField("records_sent", LongType(), True),
    StructField("eventhub_batches", LongType(), True),
    StructField("old_watermark_value", TimestampType(), True),
    StructField("new_watermark_value", TimestampType(), True),
    StructField("error_message", StringType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("finished_at", TimestampType(), True),
    StructField("created_at", TimestampType(), True),
])


def append_push_audit_rows(rows: list):
    if not rows:
        return

    def _write():
        with PUSH_AUDIT_WRITE_LOCK:
            df = spark.createDataFrame(rows, schema=PUSH_AUDIT_SCHEMA)
            df.write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)

    return with_retry("append_push_audit_rows", _write)


def append_run_audit_row(row: dict):
    def _write():
        with RUN_AUDIT_WRITE_LOCK:
            df = spark.createDataFrame([row], schema=RUN_AUDIT_SCHEMA)
            df.write.format("delta").mode("append").saveAsTable(RUN_AUDIT_TABLE)

    return with_retry("append_run_audit_row", _write)


def build_push_audit_row(
    run_id: str,
    config: dict,
    entity_id: str,
    stage: str,
    error_message: str,
    payload_json: str = None,
    record_watermark_value=None,
    batch_id: str = None
):
    return {
        "audit_id": str(uuid.uuid4()),
        "run_id": run_id,
        "config_id": config.get("config_id"),
        "layer_name": config.get("layer_name"),
        "table_schema": config.get("table_schema"),
        "table_name": config.get("table_name"),
        "entity_type": canonical_entity_type(config.get("entity_type") or config.get("table_name")),
        "entity_id": entity_id,
        "status": "FAILED",
        "failure_stage": stage,
        "error_message": str(error_message)[:4000] if error_message else None,
        "payload_json": payload_json[:8000] if payload_json else None,
        "record_watermark_value": record_watermark_value,
        "batch_id": batch_id,
        "created_at": utc_now()
    }


def audit_failure(
    run_id: str,
    config: dict,
    entity_id: str,
    stage: str,
    error_message: str,
    payload_json: str = None,
    record_watermark_value=None,
    batch_id: str = None
):
    row = build_push_audit_row(
        run_id=run_id,
        config=config,
        entity_id=entity_id,
        stage=stage,
        error_message=error_message,
        payload_json=payload_json,
        record_watermark_value=record_watermark_value,
        batch_id=batch_id
    )

    append_push_audit_rows([row])


# ============================================================
# 10. WATERMARK HELPERS
# ============================================================

def get_current_watermark(config: dict):
    watermark_id = build_watermark_id(config)

    wm_df = spark.sql(f"""
        SELECT watermark_value
        FROM {WATERMARK_TABLE}
        WHERE watermark_id = {sql_quote(watermark_id)}
        ORDER BY updated_at DESC NULLS LAST
        LIMIT 1
    """)

    rows = wm_df.collect()

    if not rows:
        return None

    return rows[0]["watermark_value"]


def init_watermark_if_missing(config: dict):
    watermark_id = build_watermark_id(config)
    watermark_col = get_watermark_col(config)

    exists_df = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM {WATERMARK_TABLE}
        WHERE watermark_id = {sql_quote(watermark_id)}
    """)

    cnt = exists_df.collect()[0]["cnt"]

    if cnt > 0:
        return

    init_schema = StructType([
        StructField("watermark_id", StringType(), True),
        StructField("watermark_col", StringType(), True),
        StructField("id_key", StringType(), True),
        StructField("watermark_value", TimestampType(), True),
        StructField("last_success_run_id", StringType(), True),
        StructField("last_success_at", TimestampType(), True),
        StructField("updated_at", TimestampType(), True),
    ])

    row = {
        "watermark_id": watermark_id,
        "watermark_col": watermark_col,
        "id_key": config["id_key"],
        "watermark_value": None,
        "last_success_run_id": None,
        "last_success_at": None,
        "updated_at": utc_now()
    }

    def _write():
        with WATERMARK_WRITE_LOCK:
            exists_inner_df = spark.sql(f"""
                SELECT COUNT(*) AS cnt
                FROM {WATERMARK_TABLE}
                WHERE watermark_id = {sql_quote(watermark_id)}
            """)
            exists_inner = exists_inner_df.collect()[0]["cnt"]

            if exists_inner > 0:
                return

            df = spark.createDataFrame([row], schema=init_schema)
            df.write.format("delta").mode("append").saveAsTable(WATERMARK_TABLE)

    with_retry("init_watermark_if_missing", _write)

    if DEBUG:
        print(f"[INIT_WATERMARK] watermark_id={watermark_id}, watermark_col={watermark_col}")


def update_watermark(config: dict, new_watermark_value, run_id: str):
    if new_watermark_value is None:
        return

    watermark_id = build_watermark_id(config)
    watermark_col = get_watermark_col(config)

    upd_schema = StructType([
        StructField("watermark_id", StringType(), True),
        StructField("watermark_col", StringType(), True),
        StructField("id_key", StringType(), True),
        StructField("watermark_value", TimestampType(), True),
        StructField("last_success_run_id", StringType(), True),
        StructField("last_success_at", TimestampType(), True),
        StructField("updated_at", TimestampType(), True),
    ])

    now_ts = utc_now()

    row = {
        "watermark_id": watermark_id,
        "watermark_col": watermark_col,
        "id_key": config["id_key"],
        "watermark_value": new_watermark_value,
        "last_success_run_id": run_id,
        "last_success_at": now_ts,
        "updated_at": now_ts
    }

    def _merge():
        with WATERMARK_WRITE_LOCK:
            tmp_view = f"tmp_wm_update_{uuid.uuid4().hex}"
            df = spark.createDataFrame([row], schema=upd_schema)
            df.createOrReplaceTempView(tmp_view)

            try:
                spark.sql(f"""
                    MERGE INTO {WATERMARK_TABLE} t
                    USING {tmp_view} s
                    ON t.watermark_id = s.watermark_id

                    WHEN MATCHED THEN UPDATE SET
                        t.watermark_col = s.watermark_col,
                        t.id_key = s.id_key,
                        t.watermark_value = s.watermark_value,
                        t.last_success_run_id = s.last_success_run_id,
                        t.last_success_at = s.last_success_at,
                        t.updated_at = s.updated_at

                    WHEN NOT MATCHED THEN INSERT (
                        watermark_id,
                        watermark_col,
                        id_key,
                        watermark_value,
                        last_success_run_id,
                        last_success_at,
                        updated_at
                    )
                    VALUES (
                        s.watermark_id,
                        s.watermark_col,
                        s.id_key,
                        s.watermark_value,
                        s.last_success_run_id,
                        s.last_success_at,
                        s.updated_at
                    )
                """)
            finally:
                try:
                    spark.catalog.dropTempView(tmp_view)
                except Exception:
                    pass

    return with_retry("update_watermark", _merge)


# ============================================================
# 11. LOAD ACTIVE CONFIGS
# ============================================================

def load_active_configs():
    df = spark.sql(f"""
        SELECT
            config_id,
            is_active,
            layer_name,
            table_schema,
            table_name,
            watermark_id,
            select_columns,
            watermark_col,
            id_key,
            COALESCE(entity_type, table_name) AS entity_type,
            COALESCE(event_type, 'upsert') AS event_type,
            COALESCE(source_name, 'fabric') AS source_name,
            COALESCE(batch_size, {GLOBAL_DEFAULT_BATCH_SIZE}) AS batch_size
        FROM {CONFIG_TABLE}
        WHERE is_active = true
    """)

    configs = [r.asDict() for r in df.collect()]

    seen = {}
    duplicates = []

    for c in configs:
        wm_id = build_watermark_id(c)

        if wm_id in seen:
            duplicates.append(wm_id)
        else:
            seen[wm_id] = True

    if duplicates:
        raise Exception(f"Duplicate active config watermark_id found: {sorted(set(duplicates))}")

    return configs


# ============================================================
# 12. READ RECORDS FROM SOURCE TABLE
# ============================================================

def read_changed_records(config: dict):
    stage = "READ_SOURCE"

    full_table_name = build_table_full_name(
        config["layer_name"],
        config["table_schema"],
        config["table_name"]
    )

    id_key = config["id_key"]
    id_keys = [k.strip() for k in str(id_key).split(",") if k.strip()]

    if not id_keys:
        raise Exception(f"[{stage}] id_key is empty for table {full_table_name}")

    select_columns = normalize_select_columns(config.get("select_columns"))
    batch_size = int(config.get("batch_size") or GLOBAL_DEFAULT_BATCH_SIZE)
    if TEST_MODE and TEST_ROWS_PER_TABLE:
        batch_size = min(batch_size, TEST_ROWS_PER_TABLE)
        print(f"[TEST_MODE] {full_table_name}: capping to {batch_size} rows")
    init_watermark_if_missing(config)
    current_wm = get_current_watermark(config)

    src_df_raw = spark.table(full_table_name)

    # Language filter (production): restrict configured entity types to English
    # rows only. Applied on the raw table before any select / watermark logic.
    if canonical_entity_type(config.get("entity_type") or config.get("table_name")) in LANG_FILTER_ENTITY_TYPES:
        if LANG_FILTER_COLUMN not in src_df_raw.columns:
            raise Exception(
                f"[{stage}] Language filter column '{LANG_FILTER_COLUMN}' not found "
                f"in table {full_table_name}"
            )
        src_df_raw = src_df_raw.where(
            F.lower(F.trim(F.col(LANG_FILTER_COLUMN).cast("string"))) == LANG_FILTER_VALUE
        )
        if DEBUG:
            print(f"[LANG_FILTER] {full_table_name}: WHERE {LANG_FILTER_COLUMN} = '{LANG_FILTER_VALUE}'")

        config_id = config.get("config_id")

        if config_id in SOFT_DELETE_CONFIG_IDS and ACTIVE_FILTER_COLUMN in src_df_raw.columns:
            soft_delete_wm_col = get_watermark_col(config)

            if not soft_delete_wm_col or soft_delete_wm_col not in src_df_raw.columns:
                raise Exception(
                    f"[{stage}] Soft-delete watermark column "
                    f"'{soft_delete_wm_col}' not found in table {full_table_name}"
                )

            is_active_bool = F.col(ACTIVE_FILTER_COLUMN).cast("boolean")
            updated_at_ts = F.col(soft_delete_wm_col).cast(TimestampType())

            if current_wm is None:
                # First run: nothing is known to exist downstream yet.
                soft_delete_eligible = F.lit(False)

            elif config_id in PRODUCT_SOFT_DELETE_CONFIG_IDS:
                # Product has no created_at:
                # send an inactive product changed since the last successful run.
                soft_delete_eligible = (
                    updated_at_ts > F.lit(current_wm).cast(TimestampType())
                )

            else:
                # POI retains the existing created_at protection.
                if SOFT_DELETE_CREATED_DATE_COLUMN not in src_df_raw.columns:
                    raise Exception(
                        f"[{stage}] Soft-delete column "
                        f"'{SOFT_DELETE_CREATED_DATE_COLUMN}' not found "
                        f"in table {full_table_name}"
                    )

                created_at_ts = F.col(
                    SOFT_DELETE_CREATED_DATE_COLUMN
                ).cast(TimestampType())

                soft_delete_eligible = (
                    created_at_ts <= F.lit(current_wm).cast(TimestampType())
                ) & (
                    updated_at_ts > F.lit(current_wm).cast(TimestampType())
                )

            src_df_raw = src_df_raw.where(
                (is_active_bool == True)
                | ((is_active_bool == False) & soft_delete_eligible)
            )

        elif ACTIVE_FILTER_COLUMN in src_df_raw.columns:
            src_df_raw = src_df_raw.where(
                F.col(ACTIVE_FILTER_COLUMN).cast("boolean")
                == ACTIVE_FILTER_VALUE
            )

    # Required non-null columns (production): drop rows whose configured
    # column(s) are NULL or blank for this entity_type (e.g. product.source_id).
    # Applied on the raw table before any select / watermark logic.
    for required_col in REQUIRED_NON_NULL_COLUMNS.get(
        canonical_entity_type(config.get("entity_type") or config.get("table_name")), []
    ):
        if required_col not in src_df_raw.columns:
            raise Exception(
                f"[{stage}] Required-non-null column '{required_col}' not found "
                f"in table {full_table_name}"
            )
        src_df_raw = src_df_raw.where(
            F.col(required_col).isNotNull()
            & (F.trim(F.col(required_col).cast("string")) != "")
        )
        if DEBUG:
            print(f"[NOT_NULL_FILTER] {full_table_name}: WHERE {required_col} IS NOT NULL/blank")

    # Required non-null columns, scoped to a specific config_id (e.g. POI must
    # have a destination_id to be published). Applied on EVERY run.
    for required_col in REQUIRED_NON_NULL_COLUMNS_BY_CONFIG.get(config.get("config_id"), []):
        if required_col not in src_df_raw.columns:
            raise Exception(
                f"[{stage}] Required-non-null column '{required_col}' not found "
                f"in table {full_table_name}"
            )
        src_df_raw = src_df_raw.where(
            F.col(required_col).isNotNull()
            & (F.trim(F.col(required_col).cast("string")) != "")
        )
        if DEBUG:
            print(f"[NOT_NULL_FILTER_BY_CONFIG] {full_table_name}: WHERE {required_col} IS NOT NULL/blank")

    # This notebook only filter POI from partner_portal ingestion source
    extra_predicate = ROW_FILTER.get(config["table_name"])
    if extra_predicate:
        src_df_raw = src_df_raw.where(extra_predicate)
        print(f"[FILTER TO PARTNER PORTAL] {full_table_name}: WHERE {extra_predicate}")
    # TEST_MODE-only extra filter, applied ON TOP of ROW_FILTER above (AND).
    # Lets you narrow further while testing (e.g. specific poi_id list and/or
    # an updated_at date range) without touching the production ROW_FILTER.
    if TEST_MODE and TEST_ROW_FILTER:
        test_extra_predicate = TEST_ROW_FILTER.get(config["table_name"])
        if test_extra_predicate:
            src_df_raw = src_df_raw.where(test_extra_predicate)
            print(f"[TEST_MODE][TEST_ROW_FILTER] {full_table_name}: WHERE {test_extra_predicate}")


    src_columns = src_df_raw.columns

    watermark_enabled = has_watermark(config)
    watermark_col = get_watermark_col(config)

    missing_id_cols = [k for k in id_keys if k not in src_columns]

    if missing_id_cols:
        raise Exception(f"[{stage}] ID key columns not found in table {full_table_name}: {missing_id_cols}")

    if watermark_enabled and watermark_col not in src_columns:
        raise Exception(f"[{stage}] Watermark column '{watermark_col}' not found in table {full_table_name}")

    is_soft_delete_config = config.get("config_id") in SOFT_DELETE_CONFIG_IDS

    if select_columns is None:
        payload_cols = list(src_columns)
    else:
        payload_cols = list(dict.fromkeys(
            select_columns
            + id_keys
            + ([watermark_col] if watermark_enabled else [])
            + ([ACTIVE_FILTER_COLUMN] if is_soft_delete_config else [])
        ))

    missing_cols = [c for c in payload_cols if c not in src_columns]

    if missing_cols:
        raise Exception(f"[{stage}] Configured columns not found in table {full_table_name}: {missing_cols}")

    # Schema-aware projection: numeric + boolean columns stay native (real JSON
    # numbers/bools); every other type is cast to STRING (original behavior).
    src_df = src_df_raw.select(*_payload_select_exprs(src_df_raw, payload_cols))

    invalid_watermark_rows = []

    if watermark_enabled:
        # Only parse values beginning with valid 4-digit date pattern.
        # This avoids parsing abnormal values like 193056-12-15...
        src_df = src_df.withColumn(
            "_safe_watermark_value",
            F.when(
                F.col(watermark_col).rlike(r"^\d{4}-\d{2}-\d{2}"),
                F.to_timestamp(F.col(watermark_col))
            ).otherwise(F.lit(None).cast(TimestampType()))
        )

        invalid_wm_df = (
            src_df
            .where(
                F.col(watermark_col).isNotNull()
                & F.col("_safe_watermark_value").isNull()
            )
            .select(
                *[F.col(k).cast("string").alias(k) for k in id_keys],
                F.col(watermark_col).alias("_invalid_watermark_value")
            )
            .limit(batch_size)
        )

        try:
            invalid_watermark_rows = [r.asDict() for r in invalid_wm_df.collect()]
        except Exception as invalid_collect_err:
            invalid_watermark_rows = [{
                "_invalid_watermark_value": None,
                "_collect_error": str(invalid_collect_err)
            }]

        src_df = src_df.where(F.col("_safe_watermark_value").isNotNull())

        if current_wm is not None:
            src_df = src_df.where(
                F.col("_safe_watermark_value") > F.lit(current_wm).cast(TimestampType())
            )

        w = Window.partitionBy(*[F.col(k) for k in id_keys]).orderBy(
            F.col("_safe_watermark_value").desc_nulls_last()
        )

        src_df = (
            src_df
            .withColumn("_rn", F.row_number().over(w))
            .where(F.col("_rn") == 1)
            .drop("_rn")
            .orderBy(F.col("_safe_watermark_value").asc())
            .limit(batch_size)
        )

    else:
        src_df = (
            src_df
            .dropDuplicates(id_keys)
            .limit(batch_size)
        )

    return src_df, current_wm, watermark_enabled, invalid_watermark_rows


# ============================================================
# 13. EVENT HUB SENDER WITH BYTES LIMITER
# ============================================================

def audit_send_failure(run_id, config, item, err, batch_id):
    try:
        audit_failure(
            run_id=run_id,
            config=config,
            entity_id=item.get("entity_id"),
            stage="PUSH_EVENTHUB_RECORD",
            error_message=str(err),
            payload_json=item.get("payload_json"),
            record_watermark_value=item.get("record_watermark_value"),
            batch_id=batch_id
        )
    except Exception as audit_err:
        print(
            f"[AUDIT_WRITE_FAILED] entity_id={item.get('entity_id')} "
            f"error={str(audit_err)[:1000]}"
        )


def send_payload_items_to_eventhub(payload_items: list, run_id: str, config: dict, batch_id: str):
    producer = EventHubProducerClient.from_connection_string(
        conn_str=EH_CONNECTION_STRING,
        eventhub_name=EH_NAME
    )

    total_sent = 0
    batch_count = 0
    failed_items = []

    def item_size_bytes(item):
        return payload_size_bytes(item["payload_json"])

    def send_one_item(item):
        nonlocal batch_count

        single_batch = producer.create_batch()
        size_bytes = item_size_bytes(item)

        try:
            single_batch.add(EventData(item["payload_json"]))
        except Exception as add_err:
            failed_items.append(item)
            audit_send_failure(run_id, config, item, add_err, batch_id)
            return False

        try:
            send_batch_with_retry(
                producer=producer,
                batch=single_batch,
                event_count=1,
                byte_count=size_bytes
            )
            batch_count += 1
            return True

        except Exception as send_err:
            failed_items.append(item)
            audit_send_failure(run_id, config, item, send_err, batch_id)
            return False

    def send_current_batch(batch, batch_items, batch_bytes):
        nonlocal total_sent, batch_count

        if not batch_items:
            return

        try:
            send_batch_with_retry(
                producer=producer,
                batch=batch,
                event_count=len(batch_items),
                byte_count=batch_bytes
            )
            total_sent += len(batch_items)
            batch_count += 1

        except Exception as batch_err:
            print(
                f"[EVENTHUB_BATCH_FAILED_SPLIT] "
                f"count={len(batch_items)} "
                f"bytes={batch_bytes} "
                f"error={str(batch_err)[:1000]}"
            )

            for single_item in batch_items:
                if send_one_item(single_item):
                    total_sent += 1

    try:
        with producer:
            batch = producer.create_batch()
            batch_items = []
            batch_bytes = 0

            for item in payload_items:
                payload_bytes = item_size_bytes(item)

                if payload_bytes > PAYLOAD_SOFT_LIMIT_BYTES:
                    failed_items.append(item)
                    audit_send_failure(
                        run_id=run_id,
                        config=config,
                        item=item,
                        err=Exception(
                            f"Payload too large before EventHub add: "
                            f"{payload_bytes} bytes > {PAYLOAD_SOFT_LIMIT_BYTES}"
                        ),
                        batch_id=batch_id
                    )
                    continue

                should_flush_by_count = len(batch_items) >= EH_MAX_EVENTS_PER_BATCH
                should_flush_by_bytes = batch_bytes + payload_bytes > EH_MAX_BYTES_PER_BATCH

                if batch_items and (should_flush_by_count or should_flush_by_bytes):
                    send_current_batch(batch, batch_items, batch_bytes)

                    batch = producer.create_batch()
                    batch_items = []
                    batch_bytes = 0

                try:
                    batch.add(EventData(item["payload_json"]))
                    batch_items.append(item)
                    batch_bytes += payload_bytes

                except ValueError:
                    send_current_batch(batch, batch_items, batch_bytes)

                    batch = producer.create_batch()
                    batch_items = []
                    batch_bytes = 0

                    try:
                        batch.add(EventData(item["payload_json"]))
                        batch_items.append(item)
                        batch_bytes += payload_bytes
                    except Exception as single_add_err:
                        failed_items.append(item)
                        audit_send_failure(run_id, config, item, single_add_err, batch_id)

            send_current_batch(batch, batch_items, batch_bytes)

    finally:
        try:
            producer.close()
        except Exception:
            pass

    return {
        "total_sent": total_sent,
        "batch_count": batch_count,
        "failed_count": len(failed_items),
        "failed_items": failed_items
    }


# ============================================================
# 14. PROCESS ONE CONFIG / ONE TABLE
# ============================================================

def process_one_config(config: dict):
    run_id = str(uuid.uuid4())
    batch_id = str(uuid.uuid4())
    started_at = utc_now()

    table_label = build_table_full_name(
        config["layer_name"],
        config["table_schema"],
        config["table_name"]
    )

    result = {
        "run_id": run_id,
        "config_id": config.get("config_id"),
        "watermark_id": build_watermark_id(config),
        "table": table_label,
        "processing_mode": get_processing_mode(config),
        "watermark_enabled": has_watermark(config),
        "status": "UNKNOWN",
        "records_read": 0,
        "records_sent": 0,
        "record_failed": 0,
        "eventhub_batches": 0,
        "old_watermark": None,
        "new_watermark": None,
        "error_stage": None,
        "error": None
    }

    current_stage = "START"

    try:
        if DEBUG:
            print(f"\n========== START TABLE: {table_label} ==========")
            print(f"config_id         = {config.get('config_id')}")
            print(f"watermark_id      = {build_watermark_id(config)}")
            print(f"processing_mode   = {get_processing_mode(config)}")
            print(f"watermark_enabled = {has_watermark(config)}")
            print(f"watermark_col     = {get_watermark_col(config)}")

        current_stage = "READ_SOURCE"
        df, old_wm, watermark_enabled, invalid_watermark_rows = read_changed_records(config)
        result["old_watermark"] = old_wm

        record_level_failed_count = 0

        if invalid_watermark_rows:
            invalid_audit_rows = []
            id_keys = [x.strip() for x in str(config["id_key"]).split(",") if x.strip()]

            for bad_row in invalid_watermark_rows:
                bad_entity_id = "|".join([str(bad_row.get(k) or "") for k in id_keys])

                invalid_audit_rows.append(build_push_audit_row(
                    run_id=run_id,
                    config=config,
                    entity_id=bad_entity_id,
                    stage="INVALID_WATERMARK_VALUE",
                    error_message=(
                        f"Invalid watermark value for column {get_watermark_col(config)}: "
                        f"{bad_row.get('_invalid_watermark_value')}. "
                        f"Record skipped, other valid records continue processing."
                    ),
                    payload_json=None,
                    record_watermark_value=None,
                    batch_id=batch_id
                ))

            try:
                append_push_audit_rows(invalid_audit_rows)
                record_level_failed_count += len(invalid_audit_rows)
            except Exception as audit_err:
                print(
                    f"[INVALID_WATERMARK_AUDIT_FAILED] table={table_label} "
                    f"error={str(audit_err)[:1000]}"
                )

        rows = df.collect()
        result["records_read"] = len(rows)

        if len(rows) == 0:
            if record_level_failed_count > 0:
                result["status"] = "PARTIAL_SUCCESS"
            else:
                result["status"] = "NO_DATA"

            result["record_failed"] = record_level_failed_count
            finished_at = utc_now()

            append_run_audit_row({
                "run_audit_id": str(uuid.uuid4()),
                "run_id": run_id,
                "config_id": config.get("config_id"),
                "layer_name": config.get("layer_name"),
                "table_schema": config.get("table_schema"),
                "table_name": config.get("table_name"),
                "status": result["status"],
                "records_read": int(result["records_read"]),
                "records_sent": int(result["records_sent"]),
                "eventhub_batches": int(result["eventhub_batches"]),
                "old_watermark_value": old_wm,
                "new_watermark_value": None,
                "error_message": (
                    f"record_level_failed_count={record_level_failed_count}; watermark not updated"
                    if record_level_failed_count > 0 else None
                ),
                "started_at": started_at,
                "finished_at": finished_at,
                "created_at": finished_at
            })

            if DEBUG:
                print(
                    f"[{result['status']}] {table_label}, "
                    f"valid_rows=0, invalid_records={record_level_failed_count}, "
                    f"current_watermark={old_wm}"
                )

            return result

        current_stage = "BUILD_PAYLOAD"

        payload_items = []
        invalid_audit_rows = []

        entity_type = canonical_entity_type(config.get("entity_type") or config["table_name"])
        event_type = config.get("event_type") or "upsert"
        source_name = config.get("source_name") or "fabric"
        # Every entity surfaces its id_key value under a uniform `id` field so the
        # KB entity is keyed consistently (EntityBase.id), regardless of the source
        # column name (product_id / destination_id / poi_id / ...). The native
        # id_key column(s) are dropped from `data` so each payload has one id field.
        id_key_cols = [k.strip() for k in str(config["id_key"]).split(",") if k.strip()]
        is_soft_delete_config = config.get("config_id") in SOFT_DELETE_CONFIG_IDS

        max_wm = None

        for row in rows:
            raw_row_dict = row.asDict(recursive=True)

            safe_wm = raw_row_dict.get("_safe_watermark_value")

            row_dict = dict(raw_row_dict)
            row_dict.pop("_safe_watermark_value", None)

            remap_destination_id(row_dict, config)
        
            # Rows that reached this point with is_active=false already passed
            # the SOFT_DELETE_FILTER (created_date > watermark), so publish
            # them as soft_delete instead of the config's normal event_type.
            row_event_type = event_type
            if is_soft_delete_config and row_dict.get(ACTIVE_FILTER_COLUMN) is False:
                row_event_type = SOFT_DELETE_EVENT_TYPE

            entity_id = build_entity_id(row_dict, config["id_key"])

            for _id_col in id_key_cols:
                row_dict.pop(_id_col, None)  # drop native id column(s)
            row_dict["id"] = entity_id  # uniform id field (product_id/destination_id/poi_id -> id)

            record_wm = safe_wm

            if watermark_enabled and safe_wm is not None:
                if max_wm is None or safe_wm > max_wm:
                    max_wm = safe_wm

            try:
                payload = build_payload(
                    row_dict=row_dict,
                    entity_type=entity_type,
                    entity_id=entity_id,
                    event_type=row_event_type,
                    source_name=source_name
                )

                # Align `data` with the Memgraph KB field contract (renames,
                # id match-key aliases, per-language id flatten, nested->JSON).
                apply_kb_field_contract(payload["data"], entity_type)

                # For the ai_enriched_poi table sent to Memgraph, use the reference
                # experience value as the payload's experience.
                if (
                    config.get("table_name") == "ai_enriched_poi"
                    and "ref_description" in payload["data"]
                ):
                    payload["data"]["description"] = payload["data"]["ref_description"]

                # Map the correct current address
                # For partner: address_line_1
                # For google: address_line_2
                if (
                    config.get("table_name") == "ai_enriched_poi"
                    and payload["data"]["ingestion_source"] == "google"
                    and "address_line_2" in payload["data"]
                ):
                    payload["data"]["address_line_1"] = payload["data"]["address_line_2"]

                payload_json = json_dumps_compact(payload)
                p_bytes = payload_size_bytes(payload_json)

            except Exception:
                invalid_audit_rows.append(build_push_audit_row(
                    run_id=run_id,
                    config=config,
                    entity_id=entity_id,
                    stage="BUILD_PAYLOAD",
                    error_message=traceback.format_exc(),
                    payload_json=None,
                    record_watermark_value=record_wm,
                    batch_id=batch_id
                ))
                continue

            current_stage = "VALIDATE_PAYLOAD_SIZE"

            if p_bytes > PAYLOAD_SOFT_LIMIT_BYTES:
                invalid_audit_rows.append(build_push_audit_row(
                    run_id=run_id,
                    config=config,
                    entity_id=entity_id,
                    stage="VALIDATE_PAYLOAD_SIZE",
                    error_message=(
                        f"Payload too large: {p_bytes} bytes > "
                        f"soft limit {PAYLOAD_SOFT_LIMIT_BYTES}"
                    ),
                    payload_json=payload_json,
                    record_watermark_value=record_wm,
                    batch_id=batch_id
                ))
                continue

            payload_items.append({
                "entity_id": entity_id,
                "payload_json": payload_json,
                "payload_bytes": p_bytes,
                "record_watermark_value": record_wm
            })

        if invalid_audit_rows:
            current_stage = "WRITE_PUSH_AUDIT"
            append_push_audit_rows(invalid_audit_rows)
            record_level_failed_count += len(invalid_audit_rows)

        if len(payload_items) == 0:
            result["record_failed"] = record_level_failed_count
            result["status"] = "PARTIAL_SUCCESS"

            finished_at = utc_now()

            append_run_audit_row({
                "run_audit_id": str(uuid.uuid4()),
                "run_id": run_id,
                "config_id": config.get("config_id"),
                "layer_name": config.get("layer_name"),
                "table_schema": config.get("table_schema"),
                "table_name": config.get("table_name"),
                "status": "PARTIAL_SUCCESS",
                "records_read": int(result["records_read"]),
                "records_sent": 0,
                "eventhub_batches": 0,
                "old_watermark_value": old_wm,
                "new_watermark_value": None,
                "error_message": (
                    "All valid payload candidates failed before Event Hub push. "
                    "Watermark not updated."
                )[:4000],
                "started_at": started_at,
                "finished_at": finished_at,
                "created_at": finished_at
            })

            return result

        if DEBUG and TEST_MODE:
            print(
                f"\n----- SAMPLE PAYLOADS (top {min(3, len(payload_items))} of "
                f"{len(payload_items)}) : entity_type={entity_type} | table={table_label} -----"
            )
            for sample_item in payload_items[:3]:
                try:
                    print(json.dumps(json.loads(sample_item["payload_json"]), ensure_ascii=False, indent=2))
                except Exception:
                    print(sample_item["payload_json"])
            print(f"----- END SAMPLE PAYLOADS : entity_type={entity_type} -----\n")

        current_stage = "PUSH_EVENTHUB"

        send_result = send_payload_items_to_eventhub(
            payload_items=payload_items,
            run_id=run_id,
            config=config,
            batch_id=batch_id
        )

        result["records_sent"] = send_result["total_sent"]
        result["eventhub_batches"] = send_result["batch_count"]

        eventhub_failed_count = send_result.get("failed_count", 0)
        record_level_failed_count += eventhub_failed_count

        current_stage = "UPDATE_WATERMARK"

        result["record_failed"] = record_level_failed_count

        if record_level_failed_count > 0:
            result["new_watermark"] = None
            result["status"] = "PARTIAL_SUCCESS"

            if DEBUG:
                print(
                    f"[PARTIAL_SUCCESS] {table_label} | "
                    f"sent={result['records_sent']} | "
                    f"failed_records={record_level_failed_count} | "
                    f"watermark_not_updated=true"
                )

        else:
            if watermark_enabled:
                new_wm = max_wm
            else:
                new_wm = utc_now()

            update_watermark(config, new_wm, run_id)

            result["new_watermark"] = new_wm
            result["status"] = "SUCCESS"

        current_stage = "WRITE_RUN_AUDIT"
        finished_at = utc_now()

        append_run_audit_row({
            "run_audit_id": str(uuid.uuid4()),
            "run_id": run_id,
            "config_id": config.get("config_id"),
            "layer_name": config.get("layer_name"),
            "table_schema": config.get("table_schema"),
            "table_name": config.get("table_name"),
            "status": result["status"],
            "records_read": int(result["records_read"]),
            "records_sent": int(result["records_sent"]),
            "eventhub_batches": int(result["eventhub_batches"]),
            "old_watermark_value": old_wm,
            "new_watermark_value": result.get("new_watermark"),
            "error_message": (
                f"record_level_failed_count={record_level_failed_count}; "
                f"watermark_updated={result.get('new_watermark') is not None}"
            )[:4000] if record_level_failed_count > 0 else None,
            "started_at": started_at,
            "finished_at": finished_at,
            "created_at": finished_at
        })

        if DEBUG:
            print(
                f"[{result['status']}] {table_label} | "
                f"read={result['records_read']} | "
                f"sent={result['records_sent']} | "
                f"failed_records={record_level_failed_count} | "
                f"eh_batches={result['eventhub_batches']} | "
                f"old_wm={old_wm} | new_wm={result.get('new_watermark')}"
            )

        return result

    except Exception as e:
        err = str(e)
        err_trace = traceback.format_exc()

        result["status"] = "FAILED"
        result["error_stage"] = current_stage
        result["error"] = err

        if DEBUG:
            print(f"[FAILED] {table_label}")
            print(f"stage={current_stage}")
            print(err)
            print(err_trace)

        try:
            audit_failure(
                run_id=run_id,
                config=config,
                entity_id=None,
                stage=current_stage,
                error_message=err_trace,
                payload_json=None,
                record_watermark_value=None,
                batch_id=batch_id
            )
        except Exception as audit_err:
            print(
                f"[AUDIT_WRITE_FAILED] table={table_label} "
                f"original_stage={current_stage} "
                f"audit_error={str(audit_err)[:1000]}"
            )

        try:
            finished_at = utc_now()
            append_run_audit_row({
                "run_audit_id": str(uuid.uuid4()),
                "run_id": run_id,
                "config_id": config.get("config_id"),
                "layer_name": config.get("layer_name"),
                "table_schema": config.get("table_schema"),
                "table_name": config.get("table_name"),
                "status": "FAILED",
                "records_read": int(result.get("records_read") or 0),
                "records_sent": int(result.get("records_sent") or 0),
                "eventhub_batches": int(result.get("eventhub_batches") or 0),
                "old_watermark_value": result.get("old_watermark"),
                "new_watermark_value": None,
                "error_message": f"[{current_stage}] {err}"[:4000],
                "started_at": started_at,
                "finished_at": finished_at,
                "created_at": finished_at
            })
        except Exception as run_audit_err:
            print(
                f"[RUN_AUDIT_WRITE_FAILED] table={table_label} "
                f"original_stage={current_stage} "
                f"run_audit_error={str(run_audit_err)[:1000]}"
            )

        return result


StatementMeta(, f327bcdb-b5e0-4259-b993-9905622f7f53, 6, Finished, Available, Finished, False)

In [5]:

# ============================================================
# 15. MAIN
# ============================================================


main_started_at = utc_now()


try:
    configs = load_active_configs()
except Exception as config_err:
    configs = []
    print(f"[CONFIG_LOAD_FAILED] {str(config_err)}")
    print(traceback.format_exc())

if not configs:
    print(f"No active config found in {CONFIG_TABLE}")
else:
    print(f"Found {len(configs)} active config(s)")

    for c in configs:
        print(
            f"- config_id={c['config_id']} | "
            f"watermark_id={build_watermark_id(c)} | "
            f"table={c['layer_name']}.{c['table_schema']}.{c['table_name']} | "
            f"processing_mode={get_processing_mode(c)} | "
            f"wm_col={get_watermark_col(c)} | "
            f"id_key={c['id_key']} | "
            f"batch_size={c['batch_size']}"
        )

results = []

# ============================================================
# Cross-entity publish ordering
# Send ALL events of one entity_type before starting the next, in this order:
#   destination -> poi -> product -> (any other entity_type, last).
# Rationale: the KB graph links children to parents (POI LOCATED_IN
# destination, product AT_POI poi), so publishing parents first shrinks the
# window where a child event is seen before its parent exists.
# NOTE: this orders *production* (each phase is fully sent before the next
# begins). Event Hub still does not guarantee cross-partition *consumption*
# order, so a downstream last-write-wins / dependency check is still advisable.
# ============================================================
ENTITY_PUBLISH_ORDER = ["destination", "poi", "product"]


def _entity_phase_key(cfg):
    et = canonical_entity_type(cfg.get("entity_type") or cfg.get("table_name"))
    try:
        return ENTITY_PUBLISH_ORDER.index(et)
    except ValueError:
        return len(ENTITY_PUBLISH_ORDER)  # unlisted entity types run last


def _run_phase_configs(phase_configs):
    # Process one phase's configs concurrently (up to MAX_PARALLEL_TABLES) and
    # append to the shared `results`. Returns only after EVERY config in the
    # phase has finished (the `with` block joins all workers), so the next
    # phase starts once all of this phase's events have been sent.
    with ThreadPoolExecutor(max_workers=MAX_PARALLEL_TABLES) as executor:
        future_map = {
            executor.submit(process_one_config, cfg): cfg
            for cfg in phase_configs
        }

        for future in as_completed(future_map):
            cfg = future_map[future]

            try:
                res = future.result()
                results.append(res)

            except Exception as e:
                err_trace = traceback.format_exc()

                fallback_result = {
                    "run_id": None,
                    "config_id": cfg.get("config_id"),
                    "watermark_id": build_watermark_id(cfg),
                    "table": f"{cfg['layer_name']}.{cfg['table_schema']}.{cfg['table_name']}",
                    "processing_mode": get_processing_mode(cfg),
                    "watermark_enabled": has_watermark(cfg),
                    "status": "FAILED",
                    "records_read": 0,
                    "records_sent": 0,
                    "record_failed": 0,
                    "eventhub_batches": 0,
                    "old_watermark": None,
                    "new_watermark": None,
                    "error_stage": "FUTURE_RESULT",
                    "error": str(e)
                }

                results.append(fallback_result)

                print(
                    f"[FUTURE_FAILED] table={fallback_result['table']} "
                    f"error={str(e)[:1000]}"
                )
                print(err_trace)


if configs:
    # Bucket configs into ordered phases by entity_type, then run each phase
    # sequentially with a barrier between phases (destination -> poi -> product).
    _phase_buckets = {}
    for cfg in configs:
        _phase_buckets.setdefault(_entity_phase_key(cfg), []).append(cfg)

    for _phase_key in sorted(_phase_buckets.keys()):
        _phase_configs = _phase_buckets[_phase_key]
        _phase_label = (
            ENTITY_PUBLISH_ORDER[_phase_key]
            if _phase_key < len(ENTITY_PUBLISH_ORDER)
            else "other"
        )
        print(
            f"\n========== PUBLISH PHASE: {_phase_label} "
            f"({len(_phase_configs)} table(s)) =========="
        )
        _run_phase_configs(_phase_configs)


# ============================================================
# 16. SUMMARY
# ============================================================

main_finished_at = utc_now()

if results:
    summary_df = spark.createDataFrame(
        [{k: ("" if v is None else str(v)) for k, v in r.items()} for r in results]
    )
    display(summary_df)

success_count = len([r for r in results if r.get("status") == "SUCCESS"])
partial_count = len([r for r in results if r.get("status") == "PARTIAL_SUCCESS"])
no_data_count = len([r for r in results if r.get("status") == "NO_DATA"])
failed_count = len([r for r in results if r.get("status") == "FAILED"])
total_count = len(results)

print("========== NOTEBOOK SUMMARY ==========")
print(f"started_at      = {main_started_at}")
print(f"finished_at     = {main_finished_at}")
print(f"total_tables    = {total_count}")
print(f"success         = {success_count}")
print(f"partial_success = {partial_count}")
print(f"no_data         = {no_data_count}")
print(f"failed          = {failed_count}")
print("======================================")

if failed_count > 0 or partial_count > 0:
    print(
        f"[WARNING] Some table(s) failed or partially succeeded. "
        f"Notebook will NOT raise exception. "
        f"Check {AUDIT_TABLE} and {RUN_AUDIT_TABLE} for details."
    )

    issue_tables = [
        {
            "table": r.get("table"),
            "config_id": r.get("config_id"),
            "status": r.get("status"),
            "stage": r.get("error_stage"),
            "error": r.get("error"),
            "record_failed": r.get("record_failed")
        }
        for r in results
        if r.get("status") in ["FAILED", "PARTIAL_SUCCESS"]
    ]

    for it in issue_tables:
        print(
            f"- {it['status']} table={it['table']} | "
            f"config_id={it['config_id']} | "
            f"stage={it['stage']} | "
            f"record_failed={it['record_failed']} | "
            f"error={str(it['error'])[:500]}"
        )
else:
    print("All table configs processed successfully.")

StatementMeta(, f327bcdb-b5e0-4259-b993-9905622f7f53, 7, Finished, Available, Finished, False)

Found 2 active config(s)
- config_id=cfg_lh_vv_gold_dbo_ai_enriched_poi | watermark_id=lh_vv_gold.dbo.ai_enriched_poi | table=lh_vv_gold.dbo.ai_enriched_poi | processing_mode=INCREMENTAL | wm_col=updated_at | id_key=poi_id | batch_size=10000
- config_id=cfg_lh_vv_gold_dbo_ai_product | watermark_id=lh_vv_gold.dbo.ai_product | table=lh_vv_gold.dbo.ai_product | processing_mode=INCREMENTAL | wm_col=updated_at | id_key=product_id | batch_size=10000

========== PUBLISH PHASE: poi (1 table(s)) ==========
[FILTER TO PARTNER PORTAL] lh_vv_gold.dbo.ai_enriched_poi: WHERE ingestion_source = 'partner_portal'

========== PUBLISH PHASE: product (1 table(s)) ==========


SynapseWidget(Synapse.DataFrame, 64e4de19-f6dd-4d03-9052-559f2d6f2b37)

========== NOTEBOOK SUMMARY ==========
started_at      = 2026-09-08 03:13:56.391161+00:00
finished_at     = 2026-09-08 03:14:36.285153+00:00
total_tables    = 2
success         = 1
partial_success = 0
no_data         = 1
failed          = 0
All table configs processed successfully.
